# B3 — Propager une rumeur, prédire une scission

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| B — Influence dans un réseau social | 3 sur 4 | 3 h | M1, M4, M7 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Les étapes B1 et B2 ont produit plusieurs mesures d'influence, qui ne s'accordent pas toujours. Ce sont des **modèles** : il faut les confronter à quelque chose de réel, ou au moins de vraisemblable.

Deux épreuves sont possibles avec ce réseau :

1. **Une rumeur.** Si une information part d'un membre, combien de membres touche-t-elle, et en combien d'étapes ? Les membres jugés « centraux » sont-ils ceux qui la diffusent le mieux ?
2. **La scission.** Zachary a noté, pour chaque membre, le camp qu'il a rejoint après la rupture. Si l'on ne connaissait que les liens d'amitié, aurait-on pu **prévoir** la scission ?

**La sous-question de cette étape : nos approches par le calcul sont-elles plausibles, et que valent-elles face à un phénomène réel ?**

**Activité** : Pour la rumeur, quelles hypothèses faut-il faire sur la façon dont elle se transmet ? Pour la scission, quelle information du réseau pourrait annoncer de quel côté chaque membre ira ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_mesure_qui_predit_la_rumeur = "degré"     # "degré", "vecteur propre", "Katz", "PageRank" ou "proximité" (modifiez)
pari_membres_mal_classes = 5                   # membres dont on prédira mal le camp (modifiez)

## ③ Abstraire

### Simuler une rumeur

On adopte un modèle simple, dit **en cascade** : à chaque étape, chaque membre qui vient d'apprendre la rumeur la transmet à chacun de ses voisins, indépendamment, avec la probabilité $p$. On répète la simulation de nombreuses fois et l'on calcule la moyenne du nombre de membres touchés (méthode de Monte-Carlo). Ce n'est plus de l'algèbre linéaire, mais une **expérience numérique** qui sert d'arbitre aux mesures algébriques.

### Couper le réseau en deux

On cherche à donner à chaque membre une valeur $x_i$, positive pour un camp et négative pour l'autre, telle que les membres liés aient des valeurs proches. Le **laplacien** du réseau, $L = D - A$, où $D$ est la matrice diagonale des degrés, mesure exactement cet écart :

$$
\mathbf{x}^{\mathsf{T}} L\,\mathbf{x} = \sum_{\text{liens } ij} \left(x_i - x_j\right)^2 .
$$

![Schéma : deux groupes de membres reliés par deux liens seulement ; le signe du deuxième vecteur propre du laplacien sépare les deux groupes](../../../assets/schemas/schema-coupe-spectrale.svg)

Le vecteur $\mathbf{1}$ donne une valeur nulle, mais ne sépare rien : c'est le vecteur propre de la valeur propre 0. Parmi les vecteurs orthogonaux à $\mathbf{1}$ et de longueur 1, celui qui rend $\mathbf{x}^{\mathsf{T}}L\mathbf{x}$ minimal est le vecteur propre de la **deuxième plus petite** valeur propre, appelé **vecteur de Fiedler**. Son signe propose une coupe.

**Activité** : Deux triangles $\{1, 2, 3\}$ et $\{4, 5, 6\}$ sont reliés par le seul lien $3$–$4$. Calculez $\mathbf{x}^{\mathsf{T}}L\mathbf{x}$ pour $\mathbf{x} = (1\,;\,1\,;\,1\,;\,-1\,;\,-1\,;\,-1)$, puis pour $\mathbf{x} = (1\,;\,-1\,;\,1\,;\,-1\,;\,1\,;\,-1)$. Laquelle de ces deux coupes le laplacien préfère-t-il ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Les mesures de l'étape B2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reseaux import charger, dessiner

membres, liens, A = charger("karate")
_, _, A_poids = charger("karate", pondere=True)
n = len(A)
valeurs, vecteurs = np.linalg.eigh(A)
lam1 = valeurs[-1]

def distances(A):
    D = np.full(A.shape, np.inf); np.fill_diagonal(D, 0)
    P = np.eye(len(A))
    for k in range(1, len(A)):
        P = P @ A
        D[(P > 0) & np.isinf(D)] = k
    return D

M = A / A.sum(axis=0)
pagerank = np.ones(n) / n
for _ in range(200):
    pagerank = 0.85 * M @ pagerank + 0.15 / n

mesures = pd.DataFrame({
    "degré": A.sum(axis=1),
    "vecteur propre": np.abs(vecteurs[:, -1]),
    "Katz": np.linalg.solve(np.eye(n) - 0.5 / lam1 * A, np.ones(n)) - 1,
    "PageRank": pagerank,
    "proximité": 1 / distances(A).mean(axis=1),
}, index=membres["id"])
mesures.head()

### Simuler la rumeur depuis chaque membre

In [ ]:
voisins = [np.nonzero(A[i])[0] for i in range(n)]

def propager(depart, p, etapes, rng):
    informes = {depart}
    nouveaux = [depart]
    for _ in range(etapes):
        suivants = []
        for i in nouveaux:
            for j in voisins[i]:
                if j not in informes and rng.random() < p:
                    informes.add(j)
                    suivants.append(j)
        nouveaux = suivants
    return len(informes)

def portee_moyenne(p, etapes=3, essais=300, graine=0):
    rng = np.random.default_rng(graine)
    return np.array([np.mean([propager(i, p, etapes, rng) for _ in range(essais)]) for i in range(n)])

portee = portee_moyenne(p=0.2)
mesures["portée (p = 0,2)"] = portee
print("Les 5 meilleurs diffuseurs (membres touchés en 3 étapes, en moyenne) :")
print(mesures["portée (p = 0,2)"].sort_values(ascending=False).head(5).round(1).to_string())

### Quelle mesure prédit le mieux la portée ?

La réponse peut dépendre de la facilité de transmission $p$.

In [ ]:
criteres = ["degré", "vecteur propre", "Katz", "PageRank", "proximité"]
tableau = {}
for p in (0.05, 0.2, 0.5):
    portee_p = pd.Series(portee_moyenne(p=p, graine=int(100 * p)), index=membres["id"])
    tableau[f"p = {p}"] = {c: mesures[c].rank().corr(portee_p.rank()) for c in criteres}
tableau = pd.DataFrame(tableau)
print("Corrélation des rangs entre chaque mesure et la portée simulée :")
print(tableau.round(2).to_string())

ax = tableau.plot.bar(figsize=(8, 3.8), color=[LIOUAI["gris"], LIOUAI["vert"], LIOUAI["navy"]], rot=0)
ax.set_ylim(0, 1); ax.set_ylabel("corrélation des rangs"); ax.grid(alpha=0.3, axis="y")
ax.set_title("Quelle mesure annonce le mieux les bons diffuseurs ?")
plt.tight_layout(); plt.show()

**Activité** : Comment la meilleure mesure change-t-elle avec $p$ ? Proposez une explication : quand la rumeur se transmet mal, qu'est-ce qui compte ? Et quand elle se transmet bien ?

Indiquez vos réponses et justifiez-les.

### Prédire la scission

In [ ]:
def fiedler(A):
    L = np.diag(A.sum(axis=1)) - A
    valeurs, vecteurs = np.linalg.eigh(L)          # valeurs croissantes ; la première vaut 0
    return valeurs, vecteurs[:, 1]

valeurs_L, v2 = fiedler(A)
camp_reel = membres["club_apres_scission"].to_numpy()
cote_hi = np.sign(v2[membres["id"].to_list().index(1)])          # le côté du professeur (membre 1)
camp_predit = np.where(np.sign(v2) == cote_hi, "Mr. Hi", "Officer")
erreurs = membres["id"][camp_predit != camp_reel].to_list()

print("Plus petites valeurs propres du laplacien :", valeurs_L[:4].round(4))
print(f"Camps bien prédits : {np.sum(camp_predit == camp_reel)} sur {n} ; membres mal classés : {erreurs}")
print(pd.crosstab(pd.Series(camp_reel, name="camp réel"), pd.Series(camp_predit, name="camp prédit")).to_string())

couleurs = [LIOUAI["vert"] if c == "Mr. Hi" else LIOUAI["jaune"] for c in camp_predit]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
dessiner(A, membres, np.ones(n), ax=axes[0], couleurs=couleurs, titre="Camps prédits par le signe du vecteur de Fiedler")
for k in np.nonzero(camp_predit != camp_reel)[0]:
    axes[0].scatter(membres["x"][k], membres["y"][k], s=700, facecolors="none", edgecolors="#C0504D", linewidths=2)
ordre = np.argsort(v2)
axes[1].bar(range(n), v2[ordre], color=[LIOUAI["vert"] if camp_reel[k] == "Mr. Hi" else LIOUAI["jaune"] for k in ordre])
axes[1].set_xticks(range(n), membres["id"][ordre], fontsize=7, rotation=90)
axes[1].axhline(0, color=LIOUAI["navy"]); axes[1].set_title("Valeurs du vecteur de Fiedler (couleur : camp réel)")
plt.tight_layout(); plt.show()

**Activité** : Combien de membres sont mal classés ? Où se trouvent-ils sur le diagramme en barres : près de zéro, ou très loin ? Que suggère leur position ?

Indiquez vos réponses et justifiez-les.

### Une hypothèse plus fine : la force des liens

Avec les poids (nombre de contextes partagés), le laplacien pondéré décrit des amitiés plus ou moins fortes.

In [ ]:
_, v2_poids = fiedler(A_poids)
cote_hi_p = np.sign(v2_poids[0])
camp_predit_p = np.where(np.sign(v2_poids) == cote_hi_p, "Mr. Hi", "Officer")
print(f"Avec les poids : {np.sum(camp_predit_p == camp_reel)} sur {n} ; mal classés : {membres['id'][camp_predit_p != camp_reel].to_list()}")

## ⑤ Interpréter

In [ ]:
meilleure = tableau["p = 0.2"].idxmax()
print(f"Votre pari : « {pari_mesure_qui_predit_la_rumeur} » ; meilleure mesure pour p = 0,2 : « {meilleure} »")
print(f"Votre pari : {pari_membres_mal_classes} membres mal classés ; résultat : {len(erreurs)}")

**Activité** :

1. L'approche par le calcul vous paraît-elle plausible pour prévoir la diffusion d'une information ? Quelles hypothèses du modèle en cascade sont les plus fragiles ?
2. Formulez une **hypothèse** qui dépasse ce que les données permettent de vérifier : par exemple, sur les raisons du choix de camp des membres mal classés. Comment pourrait-on la tester ?
3. Le calcul a-t-il été utile ? Un observateur attentif, sans calcul, aurait-il prévu la scission aussi bien ? Qu'apporte le calcul de plus (rapidité, objectivité, application à de grands réseaux…) ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Une seule épreuve.** La scission est un seul événement, sur un seul réseau. Quelle confiance accorder à un taux de réussite obtenu ainsi ?
- **Plus de deux camps.** Comment couper un réseau en trois groupes ? (Indication : les vecteurs propres suivants du laplacien.)
- **Généraliser : choisir les meilleurs relais.** Pour lancer une information, le club peut s'appuyer sur **deux** membres. Prendre les deux premiers du classement par centralité n'est pas forcément le meilleur choix : s'ils ont les mêmes amis, ils touchent les mêmes personnes. Écrivez une fonction `meilleure_paire(p, etapes)` qui choisit le premier relais comme le meilleur diffuseur seul, puis le second comme celui qui **ajoute** le plus de portée. Comparez avec la paire formée par les deux premiers selon la centralité de vecteur propre, puis avec deux membres centraux **du même camp**, par exemple les membres 1 et 3. Que concluez-vous ?

In [ ]:
def meilleure_paire(p=0.2, etapes=3, essais=300):
    # À compléter : renvoie (paire, portée moyenne)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def propager_depuis(departs, p, etapes, rng):
    informes = set(departs)
    nouveaux = list(departs)
    for _ in range(etapes):
        suivants = []
        for i in nouveaux:
            for j in voisins[i]:
                if j not in informes and rng.random() < p:
                    informes.add(j)
                    suivants.append(j)
        nouveaux = suivants
    return len(informes)

def portee_groupe(departs, p, etapes, essais=300, graine=0):
    rng = np.random.default_rng(graine)
    return np.mean([propager_depuis(departs, p, etapes, rng) for _ in range(essais)])

def meilleure_paire(p=0.2, etapes=3, essais=300):
    premier = max(range(n), key=lambda i: portee_groupe([i], p, etapes, essais))
    second = max((i for i in range(n) if i != premier), key=lambda i: portee_groupe([premier, i], p, etapes, essais))
    return (premier, second), portee_groupe([premier, second], p, etapes, essais, graine=1)

paire, portee_paire = meilleure_paire()
deux_centraux = list(np.argsort(-mesures["vecteur propre"].to_numpy())[:2])
print(f"Paire choisie pas à pas : membres {[int(membres['id'][k]) for k in paire]}, portée {portee_paire:.1f}")
print(f"Deux plus centraux      : membres {[int(membres['id'][k]) for k in deux_centraux]}, "
      f"portée {portee_groupe(deux_centraux, 0.2, 3, graine=1):.1f}")
for paire_test in ([1, 3], [34, 33]):
    indices = [membres["id"].to_list().index(m) for m in paire_test]
    print(f"Deux centraux du même camp, membres {paire_test} : portée {portee_groupe(indices, 0.2, 3, graine=1):.1f}")
print("Ici, les deux plus centraux appartiennent à deux camps différents : leurs publics se complètent.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Évaluer la plausibilité d'une approche computationnelle": 1,
    "Corroborer ses résultats": 1,
    "Formuler une hypothèse au-delà de l'étude": 1,
    "Combiner des outils pour produire le résultat": 1,
    "Juger si la pensée computationnelle est utile pour un problème réel": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "B3-propager-et-diviser", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")